In [ ]:
from tsfm_lens.chronos.circuitlens import CircuitLensChronos

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

# Get the embedding weights
embedding_weights = pipeline.model.model.shared.weight.detach().cpu().numpy()
print(f"Embedding shape: {embedding_weights.shape}")

# Perform PCA
pca = PCA()
embedding_pca = pca.fit_transform(embedding_weights)

# Create a figure with two subplots side by side
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot the first two principal components with color coding by row index
scatter = ax1.scatter(
    embedding_pca[:, 0],
    embedding_pca[:, 1],
    c=np.arange(embedding_weights.shape[0]),
    cmap="viridis",
    alpha=1,
)
ax1.set_title("PCA of Embedding Weights")
ax1.set_xlabel("Principal Component 1")
ax1.set_ylabel("Principal Component 2")
ax1.grid(True)

# Add a colorbar
cbar = fig1.colorbar(scatter, ax=ax1)
cbar.set_label("Token Index")
# Set the colorbar range from 0 to 4096
scatter.set_clim(0, 4096)

# Create scree plot
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

ax2.bar(range(1, 21), explained_variance[:20], alpha=0.7, label="Individual")
ax2.step(range(1, 21), cumulative_variance[:20], where="mid", color="red", label="Cumulative")
ax2.set_title("Scree Plot (First 20 Components)")
ax2.set_xlabel("Principal Components")
ax2.set_ylabel("Explained Variance Ratio")
ax2.set_xticks(range(1, 21))
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Create an interactive 3D plot with Plotly
df = pd.DataFrame(
    {
        "PC1": embedding_pca[:, 0],
        "PC2": embedding_pca[:, 1],
        "PC3": embedding_pca[:, 2],
        "Token Index": np.arange(embedding_weights.shape[0]),
    }
)

fig = px.scatter_3d(
    df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Token Index",
    color_continuous_scale="viridis",
    opacity=0.7,
    title="Interactive 3D PCA Visualization",
)

# Update layout for better visualization
fig.update_layout(
    scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3"),
    width=900,
    height=700,
    coloraxis_colorbar=dict(
        title="Token Index",
        tickvals=[0, 1000, 2000, 3000, 4000],
        ticktext=["0", "1000", "2000", "3000", "4000"],
    ),
)

fig.show()

# Print the variance explained by the first few components
print(f"Variance explained by first 2 components: {cumulative_variance[1]:.4f}")
print(f"Variance explained by first 5 components: {cumulative_variance[4]:.4f}")
print(f"Variance explained by first 10 components: {cumulative_variance[9]:.4f}")